# 📚 GUÍA COMPLETA DE DATAFRAMES EN SPARK

## 🎯 **OBJETIVO**
Guía exhaustiva de todos los métodos principales de DataFrames en PySpark con ejemplos prácticos.

## 📋 **CONTENIDO**
- 🔍 **Selección y Filtrado**: select, filter, where, drop, distinct
- 🔗 **Joins**: inner, left, right, full, cross joins
- 📊 **Agregaciones**: groupBy, agg, pivot, rollup, cube
- 🔄 **Transformaciones**: withColumn, dropDuplicates, union, intersect
- 🎯 **Ordenamiento**: orderBy, sort, repartition, coalesce
- 💾 **Persistencia**: cache, persist, checkpoint
- 📤 **E/S**: read, write, show, collect, take

---

## 🔧 **CONFIGURACIÓN INICIAL**


In [ ]:
# 🔄 CELDA DE REINICIO - Ejecutar si hay errores de SparkContext
try:
    if 'spark' in globals():
        print("🔄 Cerrando sesión anterior de Spark...")
        spark.stop()
        print("✅ Sesión anterior cerrada")
except:
    print("ℹ️ No había sesión anterior")

if 'spark' in globals():
    del spark

print("🚀 Listo para crear nueva sesión de Spark")


In [ ]:
# Importar todas las librerías necesarias
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import socket
import os

print("📚 Librerías importadas correctamente")


In [ ]:
# Crear SparkSession optimizada
def get_spark_master():
    try:
        hostname = socket.gethostname()
        if 'jupyter' in hostname or 'master' in hostname or 'jupyterlab' in hostname:
            return "spark://master:7077"
        else:
            return "spark://localhost:7077"
    except:
        return "local[*]"

spark_master_url = get_spark_master()
print(f"🔧 Conectando a: {spark_master_url}")

spark = SparkSession.builder \
    .appName("Guia-DataFrames-Complete") \
    .master(spark_master_url) \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.instances", "1") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

print("✅ SparkSession creada exitosamente")


In [ ]:
# Crear datos de ejemplo para todas las demostraciones
print("🏗️ Creando datasets de ejemplo...")

# Dataset 1: Empleados
empleados_data = [
    (1, "Juan Pérez", "IT", 50000, "Madrid"),
    (2, "María García", "Marketing", 45000, "Barcelona"),
    (3, "Carlos López", "IT", 55000, "Madrid"),
    (4, "Ana Martín", "HR", 40000, "Valencia"),
    (5, "Luis Rodríguez", "IT", 60000, "Sevilla"),
    (6, "Laura Sánchez", "Marketing", 48000, "Barcelona"),
    (7, "Pedro González", "Sales", 42000, "Madrid"),
    (8, "Carmen Díaz", "IT", 52000, "Valencia")
]

empleados_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("departamento", StringType(), True),
    StructField("salario", IntegerType(), True),
    StructField("ciudad", StringType(), True)
])

df_empleados = spark.createDataFrame(empleados_data, empleados_schema)

# Dataset 2: Departamentos
departamentos_data = [
    ("IT", "Tecnología", "Edificio A"),
    ("Marketing", "Comercial", "Edificio B"),
    ("HR", "Recursos Humanos", "Edificio C"),
    ("Sales", "Ventas", "Edificio A")
]

df_departamentos = spark.createDataFrame(departamentos_data, 
                                       ["codigo", "nombre_completo", "ubicacion"])

# Dataset 3: Proyectos
proyectos_data = [
    (1, "Sistema Web", "IT", 100000),
    (2, "Campaña Digital", "Marketing", 50000),
    (3, "Reclutamiento", "HR", 30000),
    (4, "CRM Nuevo", "Sales", 80000),
    (5, "App Móvil", "IT", 120000)
]

df_proyectos = spark.createDataFrame(proyectos_data,
                                   ["proyecto_id", "nombre_proyecto", "departamento", "presupuesto"])

print("✅ Datasets creados exitosamente")
print(f"📊 Empleados: {df_empleados.count()} registros")
print(f"🏢 Departamentos: {df_departamentos.count()} registros") 
print(f"📋 Proyectos: {df_proyectos.count()} registros")


## 🔍 **SECCIÓN 1: SELECCIÓN Y FILTRADO**

### **📋 Métodos principales:**
- `select()` - Seleccionar columnas
- `filter()` / `where()` - Filtrar filas
- `drop()` - Eliminar columnas  
- `distinct()` - Eliminar duplicados
- `sample()` - Muestreo aleatorio


In [ ]:
# 🔍 SELECT - Seleccionar columnas
print("=" * 60)
print("🔍 MÉTODO: select() - Seleccionar columnas específicas")
print("=" * 60)

# Seleccionar columnas específicas
df_select1 = df_empleados.select("nombre", "departamento", "salario")
print("📋 Seleccionar columnas por nombre:")
df_select1.show()

# Seleccionar con alias
df_select2 = df_empleados.select(
    col("nombre").alias("empleado"),
    col("salario").alias("sueldo_anual")
)
print("📋 Seleccionar con alias:")
df_select2.show()

# Seleccionar todas las columnas excepto una
df_select3 = df_empleados.select("*").drop("ciudad")
print("📋 Todas las columnas excepto 'ciudad':")
df_select3.show()

# Seleccionar con expresiones
df_select4 = df_empleados.select(
    col("nombre"),
    col("salario"),
    (col("salario") * 1.1).alias("salario_con_bonus")
)
print("📋 Seleccionar con expresiones calculadas:")
df_select4.show()


In [ ]:
# 🔍 FILTER/WHERE - Filtrar filas
print("=" * 60)
print("🔍 MÉTODO: filter() y where() - Filtrar filas")
print("=" * 60)

# Filter con condiciones simples
df_filter1 = df_empleados.filter(col("salario") > 50000)
print("📋 Empleados con salario > 50000:")
df_filter1.show()

# Filter con múltiples condiciones
df_filter2 = df_empleados.filter(
    (col("departamento") == "IT") & (col("salario") >= 50000)
)
print("📋 Empleados de IT con salario >= 50000:")
df_filter2.show()

# Where (sinónimo de filter)
df_where1 = df_empleados.where(col("ciudad").isin(["Madrid", "Barcelona"]))
print("📋 Empleados en Madrid o Barcelona (usando where):")
df_where1.show()

# Filter con isNull/isNotNull
df_empleados_con_nulos = df_empleados.union(
    spark.createDataFrame([(9, "Test Null", None, 30000, "Test")], empleados_schema)
)
df_filter_nulls = df_empleados_con_nulos.filter(col("departamento").isNotNull())
print("📋 Empleados con departamento no nulo:")
df_filter_nulls.show()

# Filter con like/regex
df_filter_like = df_empleados.filter(col("nombre").like("%García%"))
print("📋 Empleados con 'García' en el nombre:")
df_filter_like.show()


## 🔗 **SECCIÓN 2: JOINS**

### **📋 Tipos de joins:**
- `join()` - Inner join por defecto
- `inner` - Solo registros que coinciden en ambas tablas
- `left` - Todos los registros de la tabla izquierda
- `right` - Todos los registros de la tabla derecha
- `full` / `outer` - Todos los registros de ambas tablas
- `cross` - Producto cartesiano


In [ ]:
# 🔗 JOINS - Unir DataFrames
print("=" * 60)
print("🔗 MÉTODO: join() - Unir DataFrames")
print("=" * 60)

# Inner Join (por defecto)
df_inner = df_empleados.join(
    df_departamentos, 
    df_empleados.departamento == df_departamentos.codigo,
    "inner"
).select("nombre", "departamento", "salario", "nombre_completo", "ubicacion")

print("📋 INNER JOIN - Empleados con información de departamento:")
df_inner.show()

# Left Join
df_left = df_empleados.join(
    df_proyectos,
    df_empleados.departamento == df_proyectos.departamento,
    "left"
).select("nombre", "departamento", "salario", "nombre_proyecto", "presupuesto")

print("📋 LEFT JOIN - Empleados con proyectos (puede tener nulls):")
df_left.show()

# Right Join
df_right = df_empleados.join(
    df_proyectos,
    df_empleados.departamento == df_proyectos.departamento,
    "right"
).select("nombre", "departamento", "salario", "nombre_proyecto", "presupuesto")

print("📋 RIGHT JOIN - Proyectos con empleados (puede tener nulls):")
df_right.show()

# Full Outer Join
df_full = df_empleados.join(
    df_proyectos,
    df_empleados.departamento == df_proyectos.departamento,
    "full"
).select("nombre", "departamento", "salario", "nombre_proyecto", "presupuesto")

print("📋 FULL OUTER JOIN - Todos los registros de ambas tablas:")
df_full.show()


## 📊 **SECCIÓN 3: AGREGACIONES**

### **📋 Métodos principales:**
- `groupBy()` - Agrupar por columnas
- `agg()` - Funciones de agregación
- `pivot()` - Pivotar datos
- `rollup()` - Agregaciones jerárquicas
- `cube()` - Agregaciones multidimensionales


In [ ]:
# 📊 GROUPBY y AGG - Agregaciones
print("=" * 60)
print("📊 MÉTODO: groupBy() y agg() - Agregaciones")
print("=" * 60)

# Agregación básica por departamento
df_agg1 = df_empleados.groupBy("departamento").agg(
    count("nombre").alias("total_empleados"),
    avg("salario").alias("salario_promedio"),
    max("salario").alias("salario_maximo"),
    min("salario").alias("salario_minimo"),
    sum("salario").alias("salario_total")
)

print("📋 Estadísticas por departamento:")
df_agg1.show()

# Agregación por múltiples columnas
df_agg2 = df_empleados.groupBy("departamento", "ciudad").agg(
    count("nombre").alias("empleados_por_ciudad"),
    avg("salario").alias("salario_promedio_ciudad")
)

print("📋 Estadísticas por departamento y ciudad:")
df_agg2.show()

# Agregación con filtros
df_agg3 = df_empleados.filter(col("salario") > 45000).groupBy("departamento").agg(
    count("nombre").alias("empleados_alto_salario"),
    avg("salario").alias("salario_promedio_alto")
)

print("📋 Empleados con salario > 45000 por departamento:")
df_agg3.show()


## 🔄 **SECCIÓN 4: TRANSFORMACIONES**

### **📋 Métodos principales:**
- `withColumn()` - Agregar/modificar columnas
- `withColumnRenamed()` - Renombrar columnas
- `drop()` - Eliminar columnas
- `dropDuplicates()` - Eliminar duplicados
- `union()` - Combinar DataFrames
- `intersect()` - Intersección de DataFrames


In [ ]:
# 🔄 TRANSFORMACIONES - Modificar DataFrames
print("=" * 60)
print("🔄 MÉTODO: withColumn() - Agregar/modificar columnas")
print("=" * 60)

# Agregar columnas calculadas
df_transform1 = df_empleados.withColumn("salario_mensual", col("salario") / 12) \
                           .withColumn("categoria_salario", 
                                      when(col("salario") >= 55000, "Alto")
                                      .when(col("salario") >= 45000, "Medio")
                                      .otherwise("Bajo")) \
                           .withColumn("inicial_nombre", split(col("nombre"), " ")[0])

print("📋 DataFrame con columnas calculadas:")
df_transform1.select("nombre", "salario", "salario_mensual", "categoria_salario", "inicial_nombre").show()

# Renombrar columnas
df_transform2 = df_empleados.withColumnRenamed("nombre", "empleado") \
                           .withColumnRenamed("departamento", "depto")

print("📋 DataFrame con columnas renombradas:")
df_transform2.show()

# Eliminar duplicados
df_con_duplicados = df_empleados.union(
    spark.createDataFrame([(1, "Juan Pérez", "IT", 50000, "Madrid")], empleados_schema)
)
df_sin_duplicados = df_con_duplicados.dropDuplicates()

print("📋 DataFrame original con duplicados:")
df_con_duplicados.show()
print("📋 DataFrame sin duplicados:")
df_sin_duplicados.show()

# Union de DataFrames
nuevos_empleados = spark.createDataFrame([
    (9, "Roberto Silva", "IT", 48000, "Madrid"),
    (10, "Patricia Ruiz", "Marketing", 46000, "Barcelona")
], empleados_schema)

df_union = df_empleados.union(nuevos_empleados)
print("📋 DataFrame con nuevos empleados (union):")
df_union.show()


## 🎯 **SECCIÓN 5: ORDENAMIENTO Y PERSISTENCIA**

### **📋 Métodos principales:**
- `orderBy()` / `sort()` - Ordenar datos
- `repartition()` - Redistribuir particiones
- `coalesce()` - Reducir particiones
- `cache()` / `persist()` - Almacenar en memoria
- `unpersist()` - Liberar memoria


In [ ]:
# 🎯 ORDENAMIENTO Y PERSISTENCIA
print("=" * 60)
print("🎯 MÉTODO: orderBy() y sort() - Ordenar datos")
print("=" * 60)

# Ordenamiento simple
df_orden1 = df_empleados.orderBy("salario")
print("📋 Empleados ordenados por salario (ascendente):")
df_orden1.show()

# Ordenamiento descendente
df_orden2 = df_empleados.orderBy(col("salario").desc())
print("📋 Empleados ordenados por salario (descendente):")
df_orden2.show()

# Ordenamiento múltiple
df_orden3 = df_empleados.orderBy("departamento", col("salario").desc())
print("📋 Empleados ordenados por departamento y salario desc:")
df_orden3.show()

# Sort (sinónimo de orderBy)
df_sort = df_empleados.sort("nombre")
print("📋 Empleados ordenados por nombre (usando sort):")
df_sort.show()

print("=" * 60)
print("💾 MÉTODO: cache() y persist() - Almacenar en memoria")
print("=" * 60)

# Cache un DataFrame que usaremos múltiples veces
df_empleados_cached = df_empleados.cache()
print("✅ DataFrame cachead en memoria")

# Verificar que está cachead
print(f"📊 Número de particiones: {df_empleados_cached.rdd.getNumPartitions()}")
print(f"📊 Número de registros: {df_empleados_cached.count()}")

# Liberar memoria
df_empleados_cached.unpersist()
print("🗑️ Memoria liberada (unpersist)")


## 📤 **SECCIÓN 6: MÉTODOS DE SALIDA**

### **📋 Métodos principales:**
- `show()` - Mostrar datos en consola
- `collect()` - Obtener todos los datos como lista
- `take()` - Obtener N primeros registros
- `head()` - Obtener primeros registros
- `first()` - Obtener primer registro
- `count()` - Contar registros
- `describe()` - Estadísticas descriptivas


In [ ]:
# 📤 MÉTODOS DE SALIDA
print("=" * 60)
print("📤 MÉTODOS DE SALIDA - Obtener y mostrar datos")
print("=" * 60)

# Show con diferentes parámetros
print("📋 show() - Mostrar datos:")
df_empleados.show()

print("📋 show(5) - Mostrar solo 5 registros:")
df_empleados.show(5)

print("📋 show(3, False) - Mostrar 3 registros sin truncar:")
df_empleados.show(3, False)

# Take - Obtener N registros como lista
print("📋 take(3) - Obtener 3 registros como lista:")
registros = df_empleados.take(3)
for i, registro in enumerate(registros):
    print(f"  {i+1}: {registro}")

# Head - Primeros registros
print("📋 head(2) - Primeros 2 registros:")
primeros = df_empleados.head(2)
for registro in primeros:
    print(f"  {registro}")

# First - Primer registro
print("📋 first() - Primer registro:")
primer = df_empleados.first()
print(f"  {primer}")

# Count - Contar registros
print(f"📊 count() - Total de registros: {df_empleados.count()}")

# Describe - Estadísticas descriptivas
print("📊 describe() - Estadísticas descriptivas:")
df_empleados.describe().show()

# PrintSchema - Mostrar esquema
print("📋 printSchema() - Estructura del DataFrame:")
df_empleados.printSchema()


## 🔧 **SECCIÓN 7: FUNCIONES DE ARRAY Y ESTRUCTURAS COMPLEJAS**

### **📋 Métodos principales:**
- `explode()` - Expandir arrays en filas individuales
- `collect_list()` / `collect_set()` - Agrupar valores en arrays
- `array_contains()` - Verificar si array contiene valor
- `array_length()` - Obtener longitud del array
- `struct()` - Crear estructuras anidadas
- `split()` - Dividir strings en arrays


In [ ]:
# 🔧 FUNCIONES DE ARRAY Y ESTRUCTURAS COMPLEJAS
print("=" * 60)
print("🔧 FUNCIONES DE ARRAY Y ESTRUCTURAS COMPLEJAS")
print("=" * 60)

# Crear datos con arrays para demostración
datos_array = [
    (1, "Juan Pérez", ["Python", "Java", "Scala"], ["Madrid", "Barcelona"]),
    (2, "María García", ["R", "Python"], ["Barcelona", "Valencia"]),
    (3, "Carlos López", ["Java", "C++", "Python", "Go"], ["Madrid", "Sevilla", "Valencia"]),
    (4, "Ana Martín", ["Python", "JavaScript"], ["Valencia"]),
    (5, "Luis Rodríguez", ["Scala", "Java"], ["Sevilla", "Madrid"])
]

schema_array = StructType([
    StructField("id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("lenguajes", ArrayType(StringType()), True),
    StructField("ciudades", ArrayType(StringType()), True)
])

df_arrays = spark.createDataFrame(datos_array, schema_array)

print("📋 DataFrame con arrays:")
df_arrays.show(truncate=False)

# EXPLODE - Expandir arrays en filas
print("🔧 EXPLODE - Expandir arrays en filas individuales:")
df_explode = df_arrays.select("id", "nombre", explode("lenguajes").alias("lenguaje"))
print("📋 Lenguajes expandidos:")
df_explode.show()

# COLLECT_LIST - Agrupar valores en arrays
print("🔧 COLLECT_LIST - Agrupar valores en arrays:")
df_collect = df_explode.groupBy("nombre").agg(
    collect_list("lenguaje").alias("lenguajes_agrupados")
)
print("📋 Lenguajes agrupados por persona:")
df_collect.show(truncate=False)

# ARRAY_CONTAINS - Verificar si array contiene valor
print("🔧 ARRAY_CONTAINS - Verificar si array contiene Python:")
df_contains = df_arrays.filter(array_contains(col("lenguajes"), "Python"))
print("📋 Personas que conocen Python:")
df_contains.select("nombre", "lenguajes").show(truncate=False)

# ARRAY_LENGTH - Longitud del array
print("🔧 ARRAY_LENGTH - Longitud de arrays:")
df_length = df_arrays.withColumn("num_lenguajes", array_length(col("lenguajes"))) \
                    .withColumn("num_ciudades", array_length(col("ciudades")))
print("📋 Conteo de elementos en arrays:")
df_length.select("nombre", "num_lenguajes", "num_ciudades").show()


## 🎨 **SECCIÓN 8: FUNCIONES DE STRING AVANZADAS**

### **📋 Métodos principales:**
- `regexp_extract()` / `regexp_replace()` - Expresiones regulares
- `split()` / `concat()` - Manipulación de strings
- `translate()` / `trim()` - Limpieza de texto
- `format_string()` - Formateo avanzado
- `substring()` / `length()` - Extracción y medición
- `upper()` / `lower()` - Conversión de mayúsculas/minúsculas


In [ ]:
# 🎨 FUNCIONES DE STRING AVANZADAS
print("=" * 60)
print("🎨 FUNCIONES DE STRING AVANZADAS")
print("=" * 60)

# Crear datos con strings para demostración
datos_strings = [
    (1, "Juan Pérez", "juan.perez@empresa.com", "+34-666-123-456", "Madrid, España"),
    (2, "María García-López", "maria.garcia@empresa.com", "+34-677-234-567", "Barcelona, España"),
    (3, "Carlos López Martín", "carlos.lopez@empresa.com", "+34-688-345-678", "Sevilla, España"),
    (4, "Ana Martín-Rodríguez", "ana.martin@empresa.com", "+34-699-456-789", "Valencia, España"),
    (5, "Luis Rodríguez-Sánchez", "luis.rodriguez@empresa.com", "+34-611-567-890", "Madrid, España")
]

schema_strings = StructType([
    StructField("id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("email", StringType(), True),
    StructField("telefono", StringType(), True),
    StructField("direccion", StringType(), True)
])

df_strings = spark.createDataFrame(datos_strings, schema_strings)

print("📋 DataFrame con strings:")
df_strings.show(truncate=False)

# REGEXP_EXTRACT - Extraer usando expresiones regulares
print("🎨 REGEXP_EXTRACT - Extraer dominio del email:")
df_regex = df_strings.withColumn("dominio", regexp_extract(col("email"), r"@(.+)$", 1))
print("📋 Dominios extraídos:")
df_regex.select("nombre", "email", "dominio").show(truncate=False)

# REGEXP_REPLACE - Reemplazar usando regex
print("🎨 REGEXP_REPLACE - Limpiar formato de teléfono:")
df_replace = df_strings.withColumn("telefono_limpio", 
                                  regexp_replace(col("telefono"), r"[^\d]", ""))
print("📋 Teléfonos limpiados:")
df_replace.select("nombre", "telefono", "telefono_limpio").show(truncate=False)

# SPLIT y CONCAT - Manipulación de strings
print("🎨 SPLIT - Dividir nombres:")
df_split = df_strings.withColumn("nombre_partes", split(col("nombre"), " ")) \
                    .withColumn("primer_nombre", col("nombre_partes")[0]) \
                    .withColumn("apellidos", slice(col("nombre_partes"), 2, 10))

print("🎨 CONCAT - Concatenar strings:")
df_concat = df_split.withColumn("iniciales", 
                               concat(col("primer_nombre"), lit(" "), col("apellidos")[0]))

print("📋 Nombres divididos y concatenados:")
df_concat.select("nombre", "primer_nombre", "apellidos", "iniciales").show(truncate=False)

# TRIM, UPPER, LOWER - Limpieza de texto
print("🎨 TRIM, UPPER, LOWER - Limpieza de texto:")
df_clean = df_strings.withColumn("nombre_upper", upper(col("nombre"))) \
                    .withColumn("nombre_lower", lower(col("nombre"))) \
                    .withColumn("direccion_trim", trim(col("direccion")))

print("📋 Texto limpiado:")
df_clean.select("nombre", "nombre_upper", "nombre_lower", "direccion", "direccion_trim").show(truncate=False)

# SUBSTRING y LENGTH - Extracción y medición
print("🎨 SUBSTRING y LENGTH - Extracción y medición:")
df_substr = df_strings.withColumn("nombre_longitud", length(col("nombre"))) \
                     .withColumn("primeros_5", substring(col("nombre"), 1, 5)) \
                     .withColumn("ultimos_5", substring(col("nombre"), -5, 5))

print("📋 Extracciones de substring:")
df_substr.select("nombre", "nombre_longitud", "primeros_5", "ultimos_5").show(truncate=False)


## 📅 **SECCIÓN 9: FUNCIONES DE FECHA Y TIEMPO**

### **📋 Métodos principales:**
- `to_date()` / `to_timestamp()` - Conversiones de fecha
- `date_add()` / `date_sub()` - Aritmética de fechas
- `datediff()` / `months_between()` - Diferencias de tiempo
- `trunc()` / `date_format()` - Formateo de fechas
- `current_date()` / `current_timestamp()` - Fechas actuales
- `year()` / `month()` / `day()` - Extracción de componentes


In [ ]:
# 📅 FUNCIONES DE FECHA Y TIEMPO
print("=" * 60)
print("📅 FUNCIONES DE FECHA Y TIEMPO")
print("=" * 60)

# Crear datos con fechas para demostración
datos_fechas = [
    (1, "Juan Pérez", "2023-01-15", "2023-01-15 09:30:00", 50000),
    (2, "María García", "2022-06-20", "2022-06-20 14:45:00", 45000),
    (3, "Carlos López", "2023-03-10", "2023-03-10 08:15:00", 55000),
    (4, "Ana Martín", "2021-12-05", "2021-12-05 16:20:00", 40000),
    (5, "Luis Rodríguez", "2023-07-22", "2023-07-22 11:00:00", 60000)
]

schema_fechas = StructType([
    StructField("id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("fecha_ingreso_str", StringType(), True),
    StructField("timestamp_ingreso_str", StringType(), True),
    StructField("salario", IntegerType(), True)
])

df_fechas = spark.createDataFrame(datos_fechas, schema_fechas)

print("📋 DataFrame con fechas (strings):")
df_fechas.show(truncate=False)

# TO_DATE y TO_TIMESTAMP - Conversiones de fecha
print("📅 TO_DATE y TO_TIMESTAMP - Conversiones:")
df_convert = df_fechas.withColumn("fecha_ingreso", to_date(col("fecha_ingreso_str"), "yyyy-MM-dd")) \
                     .withColumn("timestamp_ingreso", to_timestamp(col("timestamp_ingreso_str"), "yyyy-MM-dd HH:mm:ss"))

print("📋 Fechas convertidas:")
df_convert.select("nombre", "fecha_ingreso", "timestamp_ingreso").show(truncate=False)

# DATE_ADD y DATE_SUB - Aritmética de fechas
print("📅 DATE_ADD y DATE_SUB - Aritmética de fechas:")
df_arithmetic = df_convert.withColumn("fecha_90_dias_despues", date_add(col("fecha_ingreso"), 90)) \
                         .withColumn("fecha_30_dias_antes", date_sub(col("fecha_ingreso"), 30))

print("📋 Aritmética de fechas:")
df_arithmetic.select("nombre", "fecha_ingreso", "fecha_90_dias_despues", "fecha_30_dias_antes").show(truncate=False)

# DATEDIFF y MONTHS_BETWEEN - Diferencias de tiempo
print("📅 DATEDIFF y MONTHS_BETWEEN - Diferencias:")
df_diff = df_convert.withColumn("dias_desde_ingreso", datediff(current_date(), col("fecha_ingreso"))) \
                   .withColumn("meses_desde_ingreso", months_between(current_date(), col("fecha_ingreso")))

print("📋 Diferencias de tiempo:")
df_diff.select("nombre", "fecha_ingreso", "dias_desde_ingreso", "meses_desde_ingreso").show(truncate=False)

# YEAR, MONTH, DAY - Extracción de componentes
print("📅 YEAR, MONTH, DAY - Extracción de componentes:")
df_components = df_convert.withColumn("año_ingreso", year(col("fecha_ingreso"))) \
                         .withColumn("mes_ingreso", month(col("fecha_ingreso"))) \
                         .withColumn("dia_ingreso", day(col("fecha_ingreso")))

print("📋 Componentes de fecha:")
df_components.select("nombre", "fecha_ingreso", "año_ingreso", "mes_ingreso", "dia_ingreso").show(truncate=False)

# DATE_FORMAT - Formateo de fechas
print("📅 DATE_FORMAT - Formateo de fechas:")
df_format = df_convert.withColumn("fecha_formateada", date_format(col("fecha_ingreso"), "dd/MM/yyyy")) \
                     .withColumn("fecha_larga", date_format(col("fecha_ingreso"), "EEEE, dd 'de' MMMM 'de' yyyy"))

print("📋 Fechas formateadas:")
df_format.select("nombre", "fecha_ingreso", "fecha_formateada", "fecha_larga").show(truncate=False)

# TRUNC - Truncar fechas
print("📅 TRUNC - Truncar fechas:")
df_trunc = df_convert.withColumn("mes_truncado", trunc(col("fecha_ingreso"), "MM")) \
                    .withColumn("año_truncado", trunc(col("fecha_ingreso"), "YYYY"))

print("📋 Fechas truncadas:")
df_trunc.select("nombre", "fecha_ingreso", "mes_truncado", "año_truncado").show(truncate=False)


## 🪟 **SECCIÓN 10: WINDOW FUNCTIONS AVANZADAS**

### **📋 Funciones principales:**
- `lag()` / `lead()` - Valores anteriores/siguientes
- `ntile()` - Dividir en grupos iguales
- `percent_rank()` / `cume_dist()` - Rankings porcentuales
- `dense_rank()` vs `rank()` - Diferencias de ranking
- `first_value()` / `last_value()` - Primer/último valor en ventana
- `nth_value()` - Valor en posición específica


In [ ]:
# 🪟 WINDOW FUNCTIONS AVANZADAS
print("=" * 60)
print("🪟 WINDOW FUNCTIONS AVANZADAS")
print("=" * 60)

# Usar el DataFrame de empleados para Window Functions
print("📋 DataFrame base para Window Functions:")
df_empleados.show()

# Definir ventanas
window_departamento = Window.partitionBy("departamento").orderBy(col("salario").desc())
window_global = Window.orderBy(col("salario").desc())
window_ciudad = Window.partitionBy("ciudad").orderBy(col("salario").desc())

# LAG y LEAD - Valores anteriores/siguientes
print("🪟 LAG y LEAD - Valores anteriores/siguientes:")
df_lag_lead = df_empleados.withColumn("salario_anterior", lag("salario", 1).over(window_global)) \
                         .withColumn("salario_siguiente", lead("salario", 1).over(window_global)) \
                         .withColumn("diferencia_anterior", col("salario") - col("salario_anterior"))

print("📋 Comparación con valores anteriores/siguientes:")
df_lag_lead.select("nombre", "departamento", "salario", "salario_anterior", "salario_siguiente", "diferencia_anterior").show()

# NTILE - Dividir en grupos iguales
print("🪟 NTILE - Dividir en grupos iguales:")
df_ntile = df_empleados.withColumn("cuartil_salario", ntile(4).over(window_global)) \
                      .withColumn("decil_salario", ntile(10).over(window_global))

print("📋 División en cuartiles y deciles:")
df_ntile.select("nombre", "salario", "cuartil_salario", "decil_salario").show()

# PERCENT_RANK y CUME_DIST - Rankings porcentuales
print("🪟 PERCENT_RANK y CUME_DIST - Rankings porcentuales:")
df_percent = df_empleados.withColumn("percent_rank", percent_rank().over(window_global)) \
                        .withColumn("cume_dist", cume_dist().over(window_global)) \
                        .withColumn("percentil", (percent_rank().over(window_global) * 100).cast("int"))

print("📋 Rankings porcentuales:")
df_percent.select("nombre", "salario", "percent_rank", "cume_dist", "percentil").show()

# RANK vs DENSE_RANK - Diferencias de ranking
print("🪟 RANK vs DENSE_RANK - Diferencias de ranking:")
df_rank_diff = df_empleados.withColumn("rank_salario", rank().over(window_global)) \
                          .withColumn("dense_rank_salario", dense_rank().over(window_global)) \
                          .withColumn("row_number_salario", row_number().over(window_global))

print("📋 Comparación de tipos de ranking:")
df_rank_diff.select("nombre", "salario", "rank_salario", "dense_rank_salario", "row_number_salario").show()

# FIRST_VALUE y LAST_VALUE - Primer/último valor en ventana
print("🪟 FIRST_VALUE y LAST_VALUE - Valores extremos:")
df_first_last = df_empleados.withColumn("primer_salario_depto", first_value("salario").over(window_departamento)) \
                           .withColumn("ultimo_salario_depto", last_value("salario").over(window_departamento)) \
                           .withColumn("salario_min_depto", min("salario").over(window_departamento)) \
                           .withColumn("salario_max_depto", max("salario").over(window_departamento))

print("📋 Valores extremos por departamento:")
df_first_last.select("nombre", "departamento", "salario", "primer_salario_depto", "ultimo_salario_depto", "salario_min_depto", "salario_max_depto").show()

# NTH_VALUE - Valor en posición específica
print("🪟 NTH_VALUE - Valor en posición específica:")
df_nth = df_empleados.withColumn("segundo_salario_global", nth_value("salario", 2).over(window_global)) \
                    .withColumn("tercer_salario_depto", nth_value("salario", 3).over(window_departamento))

print("📋 Valores en posiciones específicas:")
df_nth.select("nombre", "departamento", "salario", "segundo_salario_global", "tercer_salario_depto").show()


## 💾 **SECCIÓN 11: E/S DE DATOS AVANZADA**

### **📋 Formatos y opciones:**
- **Formatos**: JSON, Parquet, Avro, ORC, CSV
- **Opciones de lectura**: `option()`, `schema()`, `mode()`
- **Particionado**: `partitionBy()`, `bucketBy()`
- **Compresión**: gzip, snappy, lz4, bzip2
- **Modos de escritura**: append, overwrite, ignore, error


In [ ]:
# 💾 E/S DE DATOS AVANZADA
print("=" * 60)
print("💾 E/S DE DATOS AVANZADA")
print("=" * 60)

# Crear directorio temporal para demostración
import tempfile
import os

temp_dir = tempfile.mkdtemp()
print(f"📁 Directorio temporal: {temp_dir}")

# 1. LECTURA Y ESCRITURA DE PARQUET
print("💾 PARQUET - Formato optimizado para Spark:")
parquet_path = os.path.join(temp_dir, "empleados.parquet")

# Escribir en Parquet
df_empleados.write.mode("overwrite").parquet(parquet_path)
print("✅ DataFrame escrito en formato Parquet")

# Leer Parquet
df_parquet = spark.read.parquet(parquet_path)
print("📋 DataFrame leído desde Parquet:")
df_parquet.show(5)

# 2. LECTURA Y ESCRITURA DE JSON
print("💾 JSON - Formato legible:")
json_path = os.path.join(temp_dir, "empleados.json")

# Escribir en JSON
df_empleados.write.mode("overwrite").json(json_path)
print("✅ DataFrame escrito en formato JSON")

# Leer JSON con opciones
df_json = spark.read.option("multiline", "true").json(json_path)
print("📋 DataFrame leído desde JSON:")
df_json.show(5)

# 3. LECTURA Y ESCRITURA DE CSV CON OPCIONES
print("💾 CSV - Con opciones avanzadas:")
csv_path = os.path.join(temp_dir, "empleados.csv")

# Escribir CSV con opciones
df_empleados.write.mode("overwrite") \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("encoding", "UTF-8") \
    .csv(csv_path)
print("✅ DataFrame escrito en formato CSV con opciones")

# Leer CSV con opciones
df_csv = spark.read \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .csv(csv_path)
print("📋 DataFrame leído desde CSV:")
df_csv.show(5)

# 4. ESCRITURA PARTICIONADA
print("💾 ESCRITURA PARTICIONADA:")
partitioned_path = os.path.join(temp_dir, "empleados_particionado")

# Escribir particionado por departamento
df_empleados.write.mode("overwrite") \
    .partitionBy("departamento") \
    .parquet(partitioned_path)
print("✅ DataFrame escrito particionado por departamento")

# Ver estructura de particiones
print("📁 Estructura de particiones:")
import subprocess
try:
    result = subprocess.run(['ls', '-la', partitioned_path], capture_output=True, text=True)
    print(result.stdout)
except:
    print("No se pudo mostrar la estructura de particiones")

# 5. COMPRESIÓN
print("💾 COMPRESIÓN:")
compressed_path = os.path.join(temp_dir, "empleados_comprimido")

# Escribir con compresión gzip
df_empleados.write.mode("overwrite") \
    .option("compression", "gzip") \
    .parquet(compressed_path)
print("✅ DataFrame escrito con compresión gzip")

# 6. MODOS DE ESCRITURA
print("💾 MODOS DE ESCRITURA:")

# Crear DataFrame pequeño para demostrar modos
df_pequeno = spark.createDataFrame([(9, "Test", "IT", 30000, "Test")], empleados_schema)
test_path = os.path.join(temp_dir, "test_modos")

# OVERWRITE (por defecto)
df_pequeno.write.mode("overwrite").parquet(test_path)
print("✅ Modo OVERWRITE: Reemplaza datos existentes")

# APPEND
df_pequeno.write.mode("append").parquet(test_path)
print("✅ Modo APPEND: Agrega datos a los existentes")

# IGNORE
df_pequeno.write.mode("ignore").parquet(test_path)
print("✅ Modo IGNORE: No hace nada si el archivo existe")

# Verificar contenido final
df_test = spark.read.parquet(test_path)
print(f"📊 Registros en archivo final: {df_test.count()}")

# Limpiar archivos temporales
print(f"🗑️ Limpiando archivos temporales en: {temp_dir}")
# Nota: En un entorno real, limpiar archivos temporales


## 📊 **SECCIÓN 12: ANÁLISIS DE RENDIMIENTO**

### **📋 Herramientas de optimización:**
- `explain()` - Planes de ejecución lógicos y físicos
- `explain(mode="extended")` - Plan detallado con estadísticas
- `explain(mode="cost")` - Análisis de costos (Spark 3.0+)
- `explain(mode="formatted")` - Plan formateado para lectura
- `spark.sql.adaptive.enabled` - Optimización adaptativa
- `spark.sql.adaptive.coalescePartitions.enabled` - Coalescencia automática


In [ ]:
# 📊 ANÁLISIS DE RENDIMIENTO
print("=" * 60)
print("📊 ANÁLISIS DE RENDIMIENTO")
print("=" * 60)

# Crear una consulta compleja para analizar
print("🔍 Creando consulta compleja para análisis...")

# Consulta compleja con múltiples transformaciones
df_complejo = df_empleados \
    .join(df_departamentos, df_empleados.departamento == df_departamentos.codigo, "inner") \
    .filter(col("salario") > 45000) \
    .withColumn("salario_categoria", 
               when(col("salario") >= 55000, "Alto")
               .when(col("salario") >= 50000, "Medio-Alto")
               .otherwise("Medio")) \
    .groupBy("departamento", "salario_categoria") \
    .agg(
        count("nombre").alias("total_empleados"),
        avg("salario").alias("salario_promedio"),
        max("salario").alias("salario_maximo")
    ) \
    .orderBy("departamento", "salario_promedio")

print("📋 Resultado de la consulta compleja:")
df_complejo.show()

# 1. EXPLAIN() - Plan de ejecución básico
print("📊 EXPLAIN() - Plan de ejecución básico:")
df_complejo.explain()

# 2. EXPLAIN(MODE="EXTENDED") - Plan detallado
print("\n📊 EXPLAIN(EXTENDED) - Plan detallado:")
try:
    df_complejo.explain(mode="extended")
except:
    print("Modo 'extended' no disponible en esta versión de Spark")

# 3. EXPLAIN(MODE="FORMATTED") - Plan formateado
print("\n📊 EXPLAIN(FORMATTED) - Plan formateado:")
try:
    df_complejo.explain(mode="formatted")
except:
    print("Modo 'formatted' no disponible en esta versión de Spark")

# 4. ANÁLISIS DE PARTICIONES
print("\n📊 ANÁLISIS DE PARTICIONES:")
print(f"📊 Número de particiones del DataFrame complejo: {df_complejo.rdd.getNumPartitions()}")
print(f"📊 Número de particiones del DataFrame original: {df_empleados.rdd.getNumPartitions()}")

# 5. CACHE Y PERSISTENCIA
print("\n📊 CACHE Y PERSISTENCIA:")
print("🔄 Cacheando DataFrame para reutilización...")
df_empleados_cached = df_empleados.cache()

# Forzar evaluación del cache
df_empleados_cached.count()
print("✅ DataFrame cachead en memoria")

# Verificar si está en caché
print(f"📊 DataFrame en caché: {df_empleados_cached.storageLevel}")

# 6. CONFIGURACIÓN DE OPTIMIZACIÓN
print("\n📊 CONFIGURACIÓN DE OPTIMIZACIÓN:")
print("🔧 Configuraciones actuales de Spark:")
print(f"   • spark.sql.adaptive.enabled: {spark.conf.get('spark.sql.adaptive.enabled', 'default')}")
print(f"   • spark.sql.adaptive.coalescePartitions.enabled: {spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled', 'default')}")
print(f"   • spark.sql.adaptive.skewJoin.enabled: {spark.conf.get('spark.sql.adaptive.skewJoin.enabled', 'default')}")

# 7. ANÁLISIS DE ESTADÍSTICAS
print("\n📊 ANÁLISIS DE ESTADÍSTICAS:")
print("📈 Estadísticas del DataFrame original:")
df_empleados.describe().show()

# 8. MEJORES PRÁCTICAS DE RENDIMIENTO
print("\n📊 MEJORES PRÁCTICAS DE RENDIMIENTO:")
print("✅ Prácticas aplicadas en este notebook:")
print("   • Usar select() para limitar columnas")
print("   • Filtrar temprano con filter()")
print("   • Cachear DataFrames reutilizados")
print("   • Usar joins apropiados")
print("   • Particionar datos grandes")
print("   • Usar formatos optimizados como Parquet")

# Limpiar caché
df_empleados_cached.unpersist()
print("🗑️ Caché liberado")


## 🎯 **RESUMEN COMPLETO DE MÉTODOS DE DATAFRAMES**

### **📚 Métodos aprendidos en esta guía completa:**

#### **🔍 Sección 1: Selección y Filtrado**
- `select()` - Seleccionar columnas específicas
- `filter()` / `where()` - Filtrar filas con condiciones
- `drop()` - Eliminar columnas
- `distinct()` - Eliminar duplicados

#### **🔗 Sección 2: Joins**
- `join()` - Unir DataFrames (inner, left, right, full, cross)

#### **📊 Sección 3: Agregaciones**
- `groupBy()` - Agrupar datos
- `agg()` - Funciones de agregación (count, sum, avg, max, min)

#### **🔄 Sección 4: Transformaciones**
- `withColumn()` - Agregar/modificar columnas
- `withColumnRenamed()` - Renombrar columnas
- `dropDuplicates()` - Eliminar duplicados
- `union()` - Combinar DataFrames

#### **🎯 Sección 5: Ordenamiento y Persistencia**
- `orderBy()` / `sort()` - Ordenar datos
- `cache()` / `persist()` - Almacenar en memoria
- `unpersist()` - Liberar memoria

#### **📤 Sección 6: Métodos de Salida**
- `show()` - Mostrar datos
- `collect()` - Obtener todos los datos
- `take()` - Obtener N registros
- `count()` - Contar registros
- `describe()` - Estadísticas descriptivas

#### **🔧 Sección 7: Funciones de Array y Estructuras Complejas**
- `explode()` - Expandir arrays en filas
- `collect_list()` / `collect_set()` - Agrupar en arrays
- `array_contains()` - Verificar contenido en arrays
- `array_length()` - Longitud de arrays

#### **🎨 Sección 8: Funciones de String Avanzadas**
- `regexp_extract()` / `regexp_replace()` - Expresiones regulares
- `split()` / `concat()` - Manipulación de strings
- `trim()` / `upper()` / `lower()` - Limpieza de texto
- `substring()` / `length()` - Extracción y medición

#### **📅 Sección 9: Funciones de Fecha y Tiempo**
- `to_date()` / `to_timestamp()` - Conversiones de fecha
- `date_add()` / `date_sub()` - Aritmética de fechas
- `datediff()` / `months_between()` - Diferencias de tiempo
- `date_format()` / `trunc()` - Formateo y truncado

#### **🪟 Sección 10: Window Functions Avanzadas**
- `lag()` / `lead()` - Valores anteriores/siguientes
- `ntile()` - Dividir en grupos iguales
- `percent_rank()` / `cume_dist()` - Rankings porcentuales
- `first_value()` / `last_value()` - Valores extremos

#### **💾 Sección 11: E/S de Datos Avanzada**
- **Formatos**: Parquet, JSON, CSV con opciones
- **Particionado**: `partitionBy()` para optimización
- **Compresión**: gzip, snappy, lz4
- **Modos**: append, overwrite, ignore

#### **📊 Sección 12: Análisis de Rendimiento**
- `explain()` - Planes de ejecución
- Análisis de particiones y caché
- Configuraciones de optimización
- Mejores prácticas de rendimiento

---

## 💡 **CONSEJOS AVANZADOS PARA EL ÉXITO**

### **🎯 Mejores Prácticas Avanzadas:**
1. **Usa `select()`** para limitar columnas y mejorar rendimiento
2. **Filtra temprano** con `filter()` antes de agregaciones
3. **Cache DataFrames** que usarás múltiples veces
4. **Usa joins apropiados** según tus necesidades
5. **Optimiza con `explain()`** para entender el rendimiento
6. **Usa formatos optimizados** como Parquet para datos grandes
7. **Particiona datos** para consultas más eficientes
8. **Aprovecha Window Functions** para análisis complejos

### **🚀 Próximos pasos avanzados:**
1. **Practica** con datasets reales de gran tamaño
2. **Experimenta** con diferentes estrategias de optimización
3. **Implementa** UDFs (User Defined Functions) personalizadas
4. **Explora** Spark Streaming para datos en tiempo real
5. **Integra** con sistemas de almacenamiento distribuido

---

**🎉 ¡Has dominado TODOS los métodos principales y avanzados de DataFrames en Spark!**


## 🎯 **RESUMEN DE MÉTODOS DE DATAFRAMES**

### **📚 Métodos aprendidos en esta guía:**

#### **🔍 Selección y Filtrado:**
- `select()` - Seleccionar columnas específicas
- `filter()` / `where()` - Filtrar filas con condiciones
- `drop()` - Eliminar columnas
- `distinct()` - Eliminar duplicados

#### **🔗 Joins:**
- `join()` - Unir DataFrames (inner, left, right, full, cross)

#### **📊 Agregaciones:**
- `groupBy()` - Agrupar datos
- `agg()` - Funciones de agregación (count, sum, avg, max, min)

#### **🔄 Transformaciones:**
- `withColumn()` - Agregar/modificar columnas
- `withColumnRenamed()` - Renombrar columnas
- `dropDuplicates()` - Eliminar duplicados
- `union()` - Combinar DataFrames

#### **🎯 Ordenamiento y Persistencia:**
- `orderBy()` / `sort()` - Ordenar datos
- `cache()` / `persist()` - Almacenar en memoria
- `unpersist()` - Liberar memoria

#### **📤 Salida:**
- `show()` - Mostrar datos
- `collect()` - Obtener todos los datos
- `take()` - Obtener N registros
- `count()` - Contar registros
- `describe()` - Estadísticas descriptivas

---

## 💡 **CONSEJOS PARA EL ÉXITO**

### **🎯 Mejores Prácticas:**
1. **Usa `select()`** para limitar columnas y mejorar rendimiento
2. **Filtra temprano** con `filter()` antes de agregaciones
3. **Cache DataFrames** que usarás múltiples veces
4. **Usa joins apropiados** según tus necesidades
5. **Ordena solo cuando sea necesario** (puede ser costoso)

### **🚀 Próximos pasos:**
1. **Practica** con tus propios datos
2. **Experimenta** con diferentes combinaciones
3. **Optimiza** consultas complejas
4. **Explora** Window Functions en el tutorial avanzado

---

**🎉 ¡Has dominado los métodos principales de DataFrames en Spark!**


In [ ]:
# 🔒 Cerrar SparkSession
spark.stop()
print("🔒 SparkSession cerrada correctamente")
print("🎉 ¡Guía completa de DataFrames finalizada!")
